In [1]:
print("hello world")

hello world


#### cad specifications 

It should be sized to hold a single 608 ( ‘skate’ ) bearing, in the center of the block.

It should have counter-bored holes for M2 socket head cap screws at the corners.

The length and width of the block should be configurable by the user to any reasonable size.

In [2]:
# Check all imports work
import build123d
from build123d import *
import ocp_vscode
from ocp_vscode import show, show_object

print(f"✓ build123d version: {build123d.__version__}")
print(f"✓ ocp_vscode loaded")
print("Ready to create CAD models!")

✓ build123d version: 0.10.0
✓ ocp_vscode loaded
Ready to create CAD models!


In [1]:
from build123d import *
from ocp_vscode import show

# Body - hollow cylinder
with BuildPart() as body:
    Cylinder(radius=40, height=100)
    Cylinder(radius=36, height=95, mode=Mode.SUBTRACT)

# Lid with knob
with BuildPart() as lid:
    Cylinder(radius=42, height=8)
    with Locations((0, 0, -5)):
        Cylinder(radius=36, height=6, mode=Mode.SUBTRACT)  # Subtract to fit inside body
    with Locations((0, 0, 8)):
        Sphere(radius=10, mode=Mode.ADD)  # Knob on top

lid_moved = lid.part.move(Location((0, 0, 100)))

# Handle - 3 cylinders properly joined
with BuildPart() as handle:
    # Top horizontal cylinder (along X axis)
    with Locations((0, 0, 70)):
        Cylinder(radius=8, height=50, rotation=Rotation(0, 90, 0))
    
    # Bottom horizontal cylinder (along X axis)
    with Locations((0, 0, 30)):
        Cylinder(radius=8, height=50, rotation=Rotation(0, 90, 0))
    
    # Vertical connector (along Z axis - no rotation needed)
    # Positioned at X=25 to meet the end of horizontal cylinders (which extend from -25 to +25)
    with Locations((25, 0, 50)):
        Cylinder(radius=8, height=40)  # Default is vertical along Z

# Move handle to attach to body side
# Body radius is 40, handle extends from Y=-8 to Y=+8 (radius 8)
# Position at Y=40 + 8 = 48 to touch the body surface
handle_moved = handle.part.move(Location((-25, 48, 0)))

# Assembly
jug = Compound([body.part, lid_moved, handle_moved])

show(jug, names=["Body", "Lid", "Handle"], colors=[(0.9, 0.9, 0.95), (0.3, 0.6, 0.9), (0.7, 0.6, 0.5)])

Using port 3939
Too many names, trimming to length 1
Too many colors, trimming to length 1
+


In [2]:
L, w, t, b, h, n = 60.0, 18.0, 9.0, 0.9, 90.0, 6.0

with BuildPart() as ex29:
    with BuildSketch(Plane.XY.offset(-b)) as ex29_ow_sk:
        with BuildLine() as ex29_ow_ln:
            l1 = Line((0, 0), (0, w / 2))
            l2 = ThreePointArc(l1 @ 1, (L / 2.0, w / 2.0 + t), (L, w / 2.0))
            l3 = Line(l2 @ 1, ((l2 @ 1).X, 0, 0))
            mirror(ex29_ow_ln.line)
        make_face()
    extrude(amount=h + b)
    fillet(ex29.edges(), radius=w / 6)
    with BuildSketch(ex29.faces().sort_by(Axis.Z)[-1]):
        Circle(t)
    extrude(amount=n)
    necktopf = ex29.faces().sort_by(Axis.Z)[-1]
    offset(ex29.solids()[0], amount=-b, openings=necktopf)

show(ex29)

+
